# Week 5, Day 2: strands-agents

## What this lab covers

This lab configures an AWS Strands agent with an Azure OpenAI model and runs an asynchronous request. It then gives the agent tools for a shared SQLite task board, connects it to a local filesystem service through MCP, and combines both tool sets in a worker that plans and completes a translation task.

### Day 02: AWS Strands

In [10]:
# Necessary library imports
import os
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from strands import Agent, tool
from strands.models.openai import OpenAIModel
from openai import OpenAI
from openai import AsyncAzureOpenAI 
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters

load_dotenv(override=True)

True

In [2]:
# Let's see if the API key is working/helping us to call LLM from Azure Foundry
# From OpenAI
AZURE_OPENAI_API_KEY= os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_MODEL_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_DEPLOYMENT_GPT_41 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_41")
AZURE_OPENAI_DEPLOYMENT_GPT_54_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_54_mini")
AZURE_OPENAI_DEPLOYMENT_GPT_55 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_55")
AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini")
if AZURE_OPENAI_API_KEY:
    print("AZURE_OPENAI_API_KEY is available")
else:
    print("AZURE_OPENAI_API_KEY is not available")

# From Anthropic
AZURE_CLAUDE_DEPLOYMENT_OPUS_48=os.getenv("AZURE_CLAUDE_DEPLOYMENT_OPUS_48")
AZURE_CLAUDE_ENDPOINT=os.getenv("AZURE_CLAUDE_ENDPOINT")
AZURE_CLAUDE_API_KEY=os.getenv("AZURE_CLAUDE_API_KEY")
if AZURE_CLAUDE_API_KEY:
    print("AZURE_CLAUDE_API_KEY is avaiable")
else:
    print("AZURE_CLAUDE_API_KEY is not available")



AZURE_OPENAI_API_KEY is available
AZURE_CLAUDE_API_KEY is avaiable


#### Step 01: Create an agent

In [12]:
# let's prepare a client of openai-azure foundry
azure_openai_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)

In [13]:
# let's create a model and agent
model = OpenAIModel(
    model_id=AZURE_OPENAI_DEPLOYMENT_GPT_41,
    client=azure_openai_client
)

agent = Agent(
    model=model,
    system_prompt="You are a concise, friendly, assistant. Reply in a single short sentence."
)


#### Step 02: Run the agent

In [14]:
result =  await agent.invoke_async("Say hello in german.")

Hallo!

#### A project this week: a SQLite todo board

In [15]:
import board
board.reset_board()
board.add_goal("Read notes.txt, translate its contents into natural Spanish, and write the Spanish to Spanish.txt.")
board.list_todos()

[{'id': 1,
  'parent_id': None,
  'task': 'Read notes.txt, translate its contents into natural Spanish, and write the Spanish to Spanish.txt.',
  'status': 'pending',
  'result': ''}]

In [16]:
board.show_board()

Goal #1: Read notes.txt, translate its contents into natural Spanish, and write the Spanish to Spanish.txt.

#### Step 03: Add tools

In [ ]:
# let's define some functions for interacting with the board
@tool
def show_todos() -> list[dict]:
    """List every todo on board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

@tool
def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions.
    
    Args: 
        goal_id: The id of the goal to break down.
        steps: Short descriptions of the steps ot take, in order.    
    """
    return {"goal_id": goal_id, "steps_id": [board.add_step(goal_id, step) for step in steps]}

@tool
def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a steps or the goal) with this id as done and record a short result summary.
    
    Args: 
        task_id: The id of the todo to mark it done.
        result: a short summary of what was accomplished.
    """
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "result": result, "status": "done"}


In [18]:
complete_task

<function __main__.complete_task(task_id: int, result: str) -> dict>

In [19]:
# let's create an agent
board_agent = Agent(
    model=model,
    system_prompt="You help manage a shared todo board.",
    tools=[show_todos,complete_task]
)

# let's invoke it
result = await board_agent.invoke_async("Whatis on board right now, and what is its status..?")

tool=<<function complete_task at 0x000001C528CDF4C0>> | unrecognized tool specification



Tool #1: show_todos
Currently, there is one item on the board:

- Goal: Read notes.txt, translate its contents into natural Spanish, and write the Spanish to Spanish.txt.
- Status: Pending

No steps are listed, and this goal has not been completed yet.

#### Step 04: Add a MCP(Model Context Protocol)

In [20]:
# import some required libraries for filesystem mcp server 
from mcp.client.sse import sse_client
from strands.tools.mcp import MCPClient

filesystem_mcp = MCPClient(
    lambda: sse_client("http://127.0.0.1:8000/sse")
)

In [ ]:
# let's call the agent with above custom filesystem mcp server as tool
agent = Agent(
    model=model,
    tools = [filesystem_mcp],
    system_prompt="You can read and write files in your workspace. Use your tools to do what is asked."
)
result = await agent.invoke_async("Read notes.txt summarize it in a one short sentence.")


Tool #1: read_file
The notes outline that the team is collaboratively building a language tutor, with each member completing individual tasks from a shared board.

Error in sse_reader
Traceback (most recent call last):
  File "c:\Users\hsingh8\OneDrive - London Stock Exchange Group\Documents\Udemy Learning\Agentic Frameworks\.venv312\Lib\site-packages\httpx\_transports\default.py", line 101, in map_httpcore_exceptions
    yield
  File "c:\Users\hsingh8\OneDrive - London Stock Exchange Group\Documents\Udemy Learning\Agentic Frameworks\.venv312\Lib\site-packages\httpx\_transports\default.py", line 271, in __aiter__
    async for part in self._httpcore_stream:
  File "c:\Users\hsingh8\OneDrive - London Stock Exchange Group\Documents\Udemy Learning\Agentic Frameworks\.venv312\Lib\site-packages\httpcore\_async\connection_pool.py", line 407, in __aiter__
    raise exc from None
  File "c:\Users\hsingh8\OneDrive - London Stock Exchange Group\Documents\Udemy Learning\Agentic Frameworks\.venv312\Lib\site-packages\httpcore\_async\connection_pool.py", line 403, in __aiter__
    async for part in self._stream:
  File "c:\Users\hsingh8\OneDrive - London Stock

#### Step 05: Let's put agent in a loop with a goal

In [24]:
# let's define INSTRUCTIONS, agent with it's tools of handling filesystem and invoke it

INSTRUCTIONS="""
You're a careful worker with a todo board and a set of file tools.
Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into,
added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal.
Your files lives in the single folder, your tools are allowed to use.
"""
worker = Agent(
    model=model,
    system_prompt=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task, filesystem_mcp]
)
board.reset_board()
goal_id = board.add_goal("Read notes.txt, translate its contents into natural German, and write the German to german.txt and send an email with a summary.")
board.claim_todo(goal_id)

result=await worker.invoke_async("Please work on pending goal on the board")
board.show_board()

tool=<<function complete_task at 0x000001C528CDF4C0>> | unrecognized tool specification



Tool #1: show_todos
Pending goal:

Read notes.txt, translate its contents into natural German, and write the German to german.txt and send an email with a summary.

Plan for the goal:
1. Read notes.txt.
2. Translate its contents into natural German.
3. Write the German translation to german.txt.
4. Write a summary and send it via email.

I will now break down the goal into these concrete steps and proceed to work through them.
Tool #2: plan_steps

Tool #3: file_exists

Tool #4: read_file
Step 1: Read notes.txt — Done.

Step 2: Translate its contents into natural German — In progress.

Translation:
Welcome to the team.

Heute bauen wir gemeinsam einen kleinen Sprachtrainer, Stück für Stück. Jede helfende Person übernimmt eine einzelne Aufgabe aus dem gemeinsamen Aufgabenplan, erledigt sie sorgfältig und markiert die Aufgabe als erledigt. Lies deine Aufgabe aufmerksam, lass dir Zeit und gib dein Bestes.

I will proceed to the next step: Write the German translation to german.txt.
Tool #

Goal #1: Read notes.txt, translate its contents into natural German, and write the German to german.txt and send an email with a summary.
  Step #2: Read notes.txt
  Step #3: Translate its contents into natural German
  Step #4: Write the German translation to german.txt
  Step #5: Write a summary and send it via email